In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

# Global Configuration
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

SYSTEM_PROMPT = """
You are an expert Wall Street Financial Analyst specializing in corporate earnings.
Your task is to analyze raw financial report text and extract core business insights.

CRITICAL INSTRUCTIONS:
1. Ignore all website navigation, headers, footers, and non-financial noise.
2. Structure your response using standard Markdown headers and bullet points.
3. OUTPUT RESTRICION: Start your response directly with the Markdown content. 
   Do NOT use triple backticks (```) or wrap the text in any code block.
"""

USER_PROMPT_PREFIX = """
Please analyze the following financial report and provide a highly condensed summary.

REQUIREMENTS:
- Extract Key Financial Metrics (e.g., Revenue, EPS, Guidance) if available.
- Summarize major strategic corporate news, announcements, or product launches mentioned.
- Format the output into high-impact bullet points.

[START OF REPORT CONTENT]
"""

class WebsiteAnalyzer:
    """A unified parser and analyzer for processing financial report websites."""
    
    def __init__(self, url: str):
        self.url = url
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
        }
        # Use requests.Session for connection pooling performance
        self.session = requests.Session()
        self.soup = None
        self._fetch_and_parse()

    def _fetch_and_parse(self):
        """Fetch the website once and cache the BeautifulSoup object."""
        try:
            # Set a 10-second timeout to prevent infinite hanging
            response = self.session.get(self.url, headers=self.headers, timeout=10)
            response.raise_for_status()
            self.soup = BeautifulSoup(response.content, "html.parser")
        except requests.RequestException as e:
            print(f"Error fetching the URL: {e}")
            self.soup = None

    def get_contents(self, max_chars: int = 2000) -> str:
        """Extract and clean main body text, truncated to max_chars."""
        if not self.soup:
            return ""
            
        title = self.soup.title.string if self.soup.title else "No title found"
        body = self.soup.body
        
        if not body:
            return title
            
        # Decompose irrelevant tags to clean data
        for irrelevant in body(["script", "style", "img", "input"]):
            irrelevant.decompose()
            
        text = body.get_text(separator="\n", strip=True)
        return (title + "\n\n" + text)[:max_chars]

    def get_links(self) -> list:
        """Extract all valid hyperlinks from the cached soup."""
        if not self.soup:
            return []
        return [link.get("href") for link in self.soup.find_all("a") if link.get("href")]


def summarize_report(content: str) -> str:
    """Invoke local Ollama instance via OpenAI SDK to generate summary."""
    # Initialize the local client standard adapter
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_PREFIX + content}
    ]
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )
    return response.choices[0].message.content


def main():
    """Main execution block."""
    target_url = "https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-first-quarter-fiscal-2027"
    print("\n[INFO] Fetching and parsing website data (Single Request)...")
    
    # Analyze the website with single-fetch caching mechanism
    analyzer = WebsiteAnalyzer(target_url)
    report_content = analyzer.get_contents(max_chars=2000)
    
    if not report_content:
        print("[ERROR] Failed to retrieve website content. Exiting.")
        return
        
    print("[INFO] Submitting to Local Ollama Inference Engine...")
    summary = summarize_report(report_content)
    
    print("\n=== Financial Report Summary ===\n")
    print(summary)


if __name__ == "__main__":
    main()